In [ ]:
import sys; sys.path.insert(0, '..')
from solver.grid import Grid
from solver.mapf import Drone, MAPFSolver
from solver.scenarios import get_scenario
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np

In [ ]:
# Small 3D instance with buildings — solvable in a few seconds
grid = Grid(rows=8, cols=8, alts=4)
buildings = [
    {"row": 2, "col": 2, "height": 3},
    {"row": 5, "col": 5, "height": 3},
    {"row": 2, "col": 5, "height": 2},
    {"row": 5, "col": 2, "height": 2},
]
for b in buildings:
    grid.add_building(b["row"], b["col"], b["height"])

drones = [
    Drone(0, (0, 0, 0), (7, 7, 3)),
    Drone(1, (7, 0, 0), (0, 7, 2)),
    Drone(2, (0, 7, 0), (7, 0, 1)),
    Drone(3, (7, 7, 0), (0, 0, 3)),
]
sol = MAPFSolver(grid, drones, time_limit_s=30).solve()
print(f"Status: {sol.status} | Makespan: {sol.makespan} | Solve: {sol.solve_time_ms:.0f}ms")

In [ ]:
COLORS = ['#3b82f6','#ef4444','#22c55e','#f59e0b','#a855f7',
          '#06b6d4','#ec4899','#84cc16','#fb923c','#e11d48']

assert sol.status in ('optimal', 'feasible'), f'Solver returned: {sol.status}'

fig = plt.figure(figsize=(10, 8), facecolor='#0f172a')
ax = fig.add_subplot(111, projection='3d')
ax.set_facecolor('#0f172a')

for b in buildings:
    ax.bar3d(b["col"]-0.4, b["row"]-0.4, 0, 0.8, 0.8, b["height"],
             color='#1e3a5f', alpha=0.6, shade=True)

for i, d in enumerate(drones):
    path = sol.paths[d.id]
    xs = [p[1] for p in path]
    ys = [p[0] for p in path]
    zs = [p[2] for p in path]
    col = COLORS[i % len(COLORS)]
    ax.plot(xs, ys, zs, color=col, lw=2, alpha=0.8)
    ax.scatter([xs[0]], [ys[0]], [zs[0]], color=col, s=60, marker='o')
    ax.scatter([xs[-1]], [ys[-1]], [zs[-1]], color=col, s=100, marker='*')

ax.set_xlabel('Col', color='#94a3b8')
ax.set_ylabel('Row', color='#94a3b8')
ax.set_zlabel('Alt', color='#94a3b8')
ax.set_title(f'3D MAPF — Makespan={sol.makespan} | {sol.solve_time_ms:.0f}ms', color='white')
ax.tick_params(colors='#94a3b8')
plt.tight_layout()
plt.show()

In [ ]:
# Benchmark: solve time vs altitude layers (8x8, 3 drones)
results = []
for alts in [1, 2, 3, 4, 5]:
    g = Grid(rows=8, cols=8, alts=alts)
    ds = [
        Drone(0, (0, 0, 0), (7, 7, alts-1)),
        Drone(1, (7, 0, 0), (0, 7, alts-1)),
        Drone(2, (0, 7, 0), (7, 0, alts-1)),
    ]
    s = MAPFSolver(g, ds, time_limit_s=30).solve()
    results.append({"alts": alts, "ms": s.solve_time_ms, "status": s.status, "makespan": s.makespan})
    print(f"alts={alts}: {s.status} makespan={s.makespan} in {s.solve_time_ms:.0f}ms")

solved = [r for r in results if r["status"] in ("optimal", "feasible")]
plt.figure(facecolor='#0f172a')
ax = plt.gca()
ax.set_facecolor('#0f172a')
ax.plot([r["alts"] for r in solved], [r["ms"] for r in solved], 'o-', color='#a855f7')
ax.set_xlabel("Altitude layers", color='#94a3b8')
ax.set_ylabel("Solve time (ms)", color='#94a3b8')
ax.set_title("CP-SAT solve time vs altitude layers (8×8, 3 drones)", color='white')
ax.tick_params(colors='#94a3b8')
plt.tight_layout()
plt.show()